# Os lances de cada partida

## Demanda da aula 4

Até aqui, todas as estatísticas da Arena Analytics saíram das partidas: placar, público, fase. Mas, desde o começo do
projeto, a pasta `data/events` guarda outro conjunto de dados: o **feed de eventos**, com uma linha para cada **lance**
de cada partida (passe, disputa de bola, falta, chute, corte, escanteio e gol), com o minuto e a seleção.

O time de analytics pediu, pela primeira vez, um número que só os lances têm: **quantos passes cada seleção trocou em
cada Copa**, e quantos por partida, em média. A entrega é uma tabela que o time consulte pelo nome, como o resumo da
aula 3.

O feed é o maior volume do projeto. Por isso, a cada passo da entrega, você também vai olhar **como o Spark executa**:
o que ele lê do arquivo, quando precisa trocar dados entre partições e como junta uma tabela grande com uma pequena.

| Etapa | O que fazer | Notebook |
|---|---|---|
| Task 1 | Conferir se os gols do feed batem com os placares das partidas | `01_explain` |
| Task 2 | Trazer o ano da Copa para cada lance | `02_broadcast_join` |
| Task 3 | Calcular os passes de cada seleção em cada Copa e a média por partida | `03_shuffle_e_aqe` |
| Task 4 | Publicar a tabela `copa.passes_por_copa` e conferir o job na Spark UI | `04_tabela_e_spark_ui` |

**Pronto quando:** os gols do feed estão conferidos com os placares, e a tabela `copa.passes_por_copa` está publicada,
com os passes, as partidas e a média de passes por partida de cada seleção em cada Copa.

## Task 1: conferir se os gols do feed batem com os placares das partidas

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-lances")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"

spark

## O feed de eventos

O feed está em **Parquet**, um formato que guarda o schema junto com o dado (aula 5). `MatchID` vira `match_id`, como
nas partidas.

Uma tabela **grande** (os lances) e uma **pequena** (as partidas).

## Os gols batem com os placares?

Se o feed estiver completo, o número de lances de gol é igual à soma de `total_gols` das partidas.

## O que o Spark leu do arquivo

Quando a ação chega, o Spark enxerga o pedido inteiro (lazy evaluation, aula 2), o otimizador (**Catalyst**) reescreve
o plano, e o **plano físico** decide como executar. O `explain()` mostra o plano físico.

A árvore se lê **de baixo para cima**.

| No código | No plano |
|---|---|
| `spark.read.parquet(EVENTOS)` | `Scan parquet` |
| `.filter(F.col("event_type") == "Goal")` | `Filter` |
| `.select("match_id", "minute", "team")` | `Project` |

Nos detalhes do `Scan parquet`:

- **`ReadSchema`**: as colunas que o Spark vai realmente ler (**column pruning**).
- **`PushedFilters`**: os filtros levados para a leitura (**predicate pushdown**).

## Para consultar depois

- **Modos do `explain`:** `explain()` mostra o plano físico; `explain(True)` mostra também os planos lógicos, antes e
  depois do otimizador; `explain(mode="formatted")` separa a árvore dos detalhes.
- **A ordem não importa:** filtrar antes ou depois do `select` gera o mesmo plano, porque o otimizador reescreve a
  consulta.
- **Em SQL:** `spark.sql("EXPLAIN SELECT ...")` mostra o plano de uma consulta SQL.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()